# Probe Verification

Interactive notebook for human verification of mined probes.
Decisions are appended to `decisions.jsonl` (resume-safe).

## Rejection Criteria

Apply these consistently before starting:

| Code | Criterion | Example |
|---|---|---|
| **ambiguous** | Prompt has more than one defensible answer | Two red cups in frame, prompt says "the red cup" |
| **wrong-ground-truth** | Annotation is factually wrong | "the red cup" but the cup is clearly orange |
| **occluded-tiny** | Target or distractor is barely visible / heavily occluded | Object is <20px or >80% hidden |
| **weak-distractor** | Distractor doesn't actually tempt (too small, too far, obviously different) | Distractor is 20px wide |
| **part-color** | Attribute: color is only on a small part of the object | "the red car" but only the tail lights are red |
| **absent-item** | Negation: doubt about whether the non-wearer actually lacks the item | Hat cut off at frame edge, person might be wearing one |
| **other** | Anything else (add reason in notes) | — |

## Verification Order

1. Negation — full review (~300 distractor + 100 control, ~1.5 hr)
2. Attributes — full review (~400 distractor + 100 control, ~1.5–2 hr)
3. Spatial — light pass (~380 + 100, ~45 min)
4. Fine-grained — spot check 20% (~50 probes, ~20 min)

If any spot-check exceeds ~5% bad, escalate to full review.

In [ ]:
import json
import time
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.schema import load_probes, Probe

In [ ]:
# ── Configuration ──────────────────────────────────────────────────

# Set the probe file to verify (change per session):
PROBE_FILE = "probes/negation_distractor.json"

DECISIONS_FILE = Path("decisions.jsonl")
IMAGES_DIR = Path("data/images")

REJECT_REASONS = [
    "ambiguous",
    "wrong-ground-truth",
    "occluded-tiny",
    "weak-distractor",
    "part-color",
    "absent-item",
    "other",
]

In [ ]:
# ── Load probes and existing decisions ─────────────────────────────

probes = load_probes(PROBE_FILE)
print(f"Loaded {len(probes)} probes from {PROBE_FILE}")

# Sort by pair_id so mirrors are shown consecutively
probes.sort(key=lambda p: (p.pair_id or "", p.probe_id))

# Load existing decisions
decided: dict[str, dict] = {}
if DECISIONS_FILE.exists():
    with DECISIONS_FILE.open() as f:
        for line in f:
            line = line.strip()
            if line:
                d = json.loads(line)
                decided[d["probe_id"]] = d

# Filter to undecided probes
undecided = [p for p in probes if p.probe_id not in decided]
print(f"Already decided: {len(decided)}")
print(f"Remaining: {len(undecided)}")

In [ ]:
# ── Image path resolver ────────────────────────────────────────────

def get_image_path(probe: Probe) -> Path | None:
    """Resolve probe to its downloaded image path."""
    if probe.image_source in ("coco_train2017", "lvis_v1_train"):
        return IMAGES_DIR / "coco" / f"{int(probe.image_id):012d}.jpg"
    elif probe.image_source == "visual_genome":
        vg_dir = IMAGES_DIR / "vg"
        for ext in ("jpg", "png", "jpeg"):
            p = vg_dir / f"{probe.image_id}.{ext}"
            if p.exists():
                return p
    return None

In [ ]:
# ── Visualization ─────────────────────────────────────────────────

def show_probe(probe: Probe, ax=None):
    """Display image with target (green) and distractor (red) boxes."""
    img_path = get_image_path(probe)
    if img_path is None or not img_path.exists():
        print(f"Image not found for {probe.probe_id}")
        return

    img = Image.open(img_path)
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)

    # Target box (green)
    x1, y1, x2, y2 = probe.target_box
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=3, edgecolor="lime", facecolor="none", label="target")
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, "TARGET", color="lime", fontsize=10,
            fontweight="bold", backgroundcolor="black")

    # Distractor box (red)
    if probe.distractor_box is not None:
        x1, y1, x2, y2 = probe.distractor_box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=3, edgecolor="red", facecolor="none", label="distractor")
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, "DISTRACTOR", color="red", fontsize=10,
                fontweight="bold", backgroundcolor="black")

    ax.set_title(probe.prompt, fontsize=18, fontweight="bold", pad=15)
    ax.set_xlabel(f"{probe.phenomenon}  |  {probe.probe_id}  |  pair: {probe.pair_id}",
                  fontsize=9, color="gray")
    ax.axis("off")
    plt.tight_layout()
    return ax

In [ ]:
# ── Interactive verifier ───────────────────────────────────────────

class ProbeVerifier:
    def __init__(self, probes: list[Probe], decisions_path: Path):
        self.probes = probes
        self.decisions_path = decisions_path
        self.index = 0

        # Widgets
        self.output = widgets.Output()
        self.reason_dropdown = widgets.Dropdown(
            options=REJECT_REASONS,
            value="ambiguous",
            description="Reason:",
            layout=widgets.Layout(width="250px"),
        )
        self.notes_input = widgets.Text(
            placeholder="Optional notes",
            description="Notes:",
            layout=widgets.Layout(width="400px"),
        )
        self.keep_btn = widgets.Button(
            description="Keep",
            button_style="success",
            layout=widgets.Layout(width="100px"),
        )
        self.reject_btn = widgets.Button(
            description="Reject",
            button_style="danger",
            layout=widgets.Layout(width="100px"),
        )
        self.flag_btn = widgets.Button(
            description="Flag",
            button_style="warning",
            layout=widgets.Layout(width="100px"),
        )
        self.skip_btn = widgets.Button(
            description="Skip",
            button_style="",
            layout=widgets.Layout(width="100px"),
        )
        self.progress_label = widgets.HTML()

        self.keep_btn.on_click(lambda _: self._decide("keep"))
        self.reject_btn.on_click(lambda _: self._decide("reject"))
        self.flag_btn.on_click(lambda _: self._decide("flag"))
        self.skip_btn.on_click(lambda _: self._advance())

        buttons = widgets.HBox([self.keep_btn, self.reject_btn,
                                self.flag_btn, self.skip_btn])
        controls = widgets.VBox([buttons,
                                 widgets.HBox([self.reason_dropdown,
                                               self.notes_input]),
                                 self.progress_label])

        display(controls)
        display(self.output)
        self._show_current()

    def _decide(self, decision: str):
        if self.index >= len(self.probes):
            return
        probe = self.probes[self.index]
        record = {
            "probe_id": probe.probe_id,
            "decision": decision,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "source_file": PROBE_FILE,
        }
        if decision == "reject":
            record["reason"] = self.reason_dropdown.value
        if self.notes_input.value.strip():
            record["notes"] = self.notes_input.value.strip()

        with self.decisions_path.open("a") as f:
            f.write(json.dumps(record) + "\n")

        self.notes_input.value = ""
        self._advance()

    def _advance(self):
        self.index += 1
        self._show_current()

    def _show_current(self):
        with self.output:
            clear_output(wait=True)
            if self.index >= len(self.probes):
                print("All probes reviewed!")
                self._update_progress()
                return
            probe = self.probes[self.index]
            print(f"\n[{self.index + 1}/{len(self.probes)}]  "
                  f"probe_id={probe.probe_id}  pair={probe.pair_id}")
            if probe.notes:
                print(f"  notes: {probe.notes}")
            show_probe(probe)
            plt.show()
        self._update_progress()

    def _update_progress(self):
        # Re-read decisions to get live counts
        counts: dict[str, int] = Counter()
        if self.decisions_path.exists():
            with self.decisions_path.open() as f:
                for line in f:
                    line = line.strip()
                    if line:
                        d = json.loads(line)
                        counts[d["decision"]] += 1

        remaining = len(self.probes) - self.index
        self.progress_label.value = (
            f"<b>Progress:</b> "
            f"<span style='color:green'>Keep: {counts.get('keep', 0)}</span> | "
            f"<span style='color:red'>Reject: {counts.get('reject', 0)}</span> | "
            f"<span style='color:orange'>Flag: {counts.get('flag', 0)}</span> | "
            f"Remaining: {remaining}"
        )

In [ ]:
# ── Start verification ─────────────────────────────────────────────
# Change PROBE_FILE above, re-run the load cell, then run this cell.

verifier = ProbeVerifier(undecided, DECISIONS_FILE)

---
## Summary Dashboard

Run this cell at any time to see live counts per phenomenon.

In [ ]:
# ── Summary: kept/rejected/remaining per phenomenon ───────────────

def show_summary():
    # Load all decisions
    decisions: dict[str, dict] = {}
    if DECISIONS_FILE.exists():
        with DECISIONS_FILE.open() as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    decisions[d["probe_id"]] = d

    # Load all probe files
    all_probes: list[Probe] = []
    for pf in sorted(Path("probes").glob("*.json")):
        if "extras" in pf.name:
            continue
        try:
            all_probes.extend(load_probes(pf))
        except Exception:
            pass

    # Count per phenomenon
    by_phenom: dict[str, dict[str, int]] = defaultdict(
        lambda: {"keep": 0, "reject": 0, "flag": 0, "undecided": 0})
    reject_reasons: Counter = Counter()

    for p in all_probes:
        d = decisions.get(p.probe_id)
        if d is None:
            by_phenom[p.phenomenon]["undecided"] += 1
        else:
            by_phenom[p.phenomenon][d["decision"]] += 1
            if d["decision"] == "reject" and "reason" in d:
                reject_reasons[d["reason"]] += 1

    print(f"{'Phenomenon':<30} {'Keep':>6} {'Reject':>8} {'Flag':>6} {'Undecided':>10} {'Total':>7}")
    print("-" * 75)
    totals = {"keep": 0, "reject": 0, "flag": 0, "undecided": 0}
    for phenom in sorted(by_phenom.keys()):
        c = by_phenom[phenom]
        total = sum(c.values())
        print(f"{phenom:<30} {c['keep']:>6} {c['reject']:>8} {c['flag']:>6} {c['undecided']:>10} {total:>7}")
        for k in totals:
            totals[k] += c[k]
    total_all = sum(totals.values())
    print("-" * 75)
    print(f"{'TOTAL':<30} {totals['keep']:>6} {totals['reject']:>8} {totals['flag']:>6} {totals['undecided']:>10} {total_all:>7}")

    if reject_reasons:
        print(f"\nRejection reasons:")
        for reason, count in reject_reasons.most_common():
            print(f"  {reason:<25} {count}")

show_summary()

---
## Re-Verification Sample (10%)

After primary verification, run this cell to pick a random 10% sample
for a teammate to re-verify independently. Then compute agreement.

In [ ]:
import random

def pick_reverify_sample(seed: int = 123, fraction: float = 0.10):
    """Pick a random sample of decided probes for re-verification."""
    decisions: dict[str, dict] = {}
    if DECISIONS_FILE.exists():
        with DECISIONS_FILE.open() as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    decisions[d["probe_id"]] = d

    decided_ids = sorted(decisions.keys())
    n = max(1, int(len(decided_ids) * fraction))
    rng = random.Random(seed)
    sample = rng.sample(decided_ids, min(n, len(decided_ids)))

    reverify_path = Path("reverify_sample.json")
    with reverify_path.open("w") as f:
        json.dump(sample, f, indent=2)
    print(f"Picked {len(sample)} probes for re-verification → {reverify_path}")
    print(f"Teammate uses the same notebook with these probe_ids.")
    return sample


def compute_agreement(
    primary_path: Path = DECISIONS_FILE,
    reverify_path: Path = Path("reverify_decisions.jsonl"),
):
    """Compute agreement between primary and re-verification decisions."""
    primary: dict[str, str] = {}
    with primary_path.open() as f:
        for line in f:
            d = json.loads(line.strip())
            primary[d["probe_id"]] = d["decision"]

    reverify: dict[str, str] = {}
    with reverify_path.open() as f:
        for line in f:
            d = json.loads(line.strip())
            reverify[d["probe_id"]] = d["decision"]

    common = set(primary.keys()) & set(reverify.keys())
    if not common:
        print("No overlapping probe_ids found.")
        return

    agree = sum(1 for pid in common if primary[pid] == reverify[pid])
    pct = agree / len(common) * 100
    print(f"Agreement: {agree}/{len(common)} = {pct:.1f}%")

    # Disagreements
    disagree = [(pid, primary[pid], reverify[pid])
                for pid in sorted(common) if primary[pid] != reverify[pid]]
    if disagree:
        print(f"\nDisagreements ({len(disagree)}):")
        for pid, p, r in disagree[:20]:
            print(f"  {pid}: primary={p}, reverify={r}")

# Uncomment to pick sample:
# pick_reverify_sample()

# Uncomment after teammate finishes:
# compute_agreement()